# Q-Table Generation

## Imports

In [ ]:
from google.colab import drive
import os
import numpy as np
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import time as t
import json
import sys
import ast
from tqdm import tqdm
import random
import seaborn as sns

## Drive Setup

In [ ]:
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [ ]:
city_name = "Columbus"
mini = True

In [ ]:
personal_dir = "./gdrive/MyDrive/DeliverAI Data Folder/"
data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"

# See directory content
data = os.listdir(data_dir)
print("Files in directory : ", data)

Files in directory :  ['Census Data', 'Original Location Data', 'Processed Location Data', 'Images', 'Hotspot Data', 'Q Tables', 'avg_hotspot_data.json', 'ExecutionTime_7_31.txt', 'UMST Graph', 'Deliveries', 'Graphing']


In [ ]:
# #For DOCKER use
# personal_dir = "./DeliverAI Data Folder/"
# data_dir = f"{personal_dir}{city_name}_mini - RL Delivery Data" if mini else f"{personal_dir}{city_name} - RL Delivery Data"

# # See directory content
# data = os.listdir(data_dir)
# print("Files in directory : ", data)

### Variables

In [ ]:
superspot_hotspot_ratio = 1  # Change this as necessary
min_children = superspot_hotspot_ratio - 0  # Do not change

## Load Data

In [ ]:
# Import census data
census_df = gpd.read_file(data_dir + "/Census Data/census_tract_data.geojson")

# Import distance/time data from graphhopper
distance_matrix = np.load(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/distance_adjacency_matrix.npy")
time_matrix = np.load(data_dir + f"/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/time_adjacency_matrix.npy")

gh_df = pd.read_csv(data_dir + f'/Hotspot Data/Data-{min_children}-{superspot_hotspot_ratio}/graphhopper_dataframe.csv', dtype={'GEOID': 'string'}).drop(["Unnamed: 0"], axis=1)
gh_df['children'] = gh_df['children'].apply(ast.literal_eval)

# Selected tract geoid list
selected_tracts = gh_df['GEOID'].tolist()
print(len(selected_tracts))

# VERY IMPORTANT! THIS AFFECTS HOW THE MODEL WORKS, MUST BE THE SAME INDEXES AS WHEN GENERATED!
with open(data_dir + "/Hotspot Data/map_geoid_index.json", 'r') as f:
    tract_to_index = json.load(f)
index_to_tract = {v: k for k, v in tract_to_index.items()}

FileNotFoundError: [Errno 2] No such file or directory: './gdrive/MyDrive/DeliverAI Data Folder/Columbus_mini - RL Delivery Data/Hotspot Data/Data-1-1/distance_adjacency_matrix.npy'

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 500)
pd.set_option('display.colheader_justify', 'center')
pd.set_option('display.precision', 2)

def print_matrix(matrix):
    modified_matrix = np.where(matrix > 1000000, -1, matrix)

    index_labels = [index_to_tract[i][-4:] for i in range(len(matrix))]  # Setting row labels
    columns_labels = [index_to_tract[i][-4:] for i in range(len(matrix))]  # Setting col labels

    print(pd.DataFrame(modified_matrix, columns=columns_labels, index=index_labels))


In [ ]:
print_matrix(time_matrix)

In [ ]:
def make_plottable(arr, threshold=sys.maxsize):
    arr = np.where(arr == threshold, -1, arr)
    return arr

def filter_above_threshold(arr1, arr2, threshold):
    mask = (arr1.flatten() <= threshold) & (arr2.flatten() <= threshold)
    return arr1.flatten()[mask], arr2.flatten()[mask]

threshold = 1000000

filtered_distance, filtered_time = filter_above_threshold(make_plottable(distance_matrix), make_plottable(time_matrix), threshold)

plt.scatter(filtered_distance, filtered_time)
plt.show()

In [ ]:
# num_children_list = [len(l) for l in list(gh_df['children']) if len(l) != 0]



# sns.histplot(num_children_list, binwidth=10)
# plt.xlabel("Number of Children in Group")
# plt.ylabel("Frequency")
# plt.title(f"[{city_name}] Histogram of Frequency of Number of Children in Group")
# plt.show()

## Normalize Matrices

In [ ]:
nodes = len(selected_tracts)
delta = 0

D = np.full((nodes,nodes),-1.0)
T = np.full((nodes,nodes),-1.0)

# All this stuff below is to just min-max normalize the matrix.
min_distance = np.min(distance_matrix)
max_distance = np.max(distance_matrix)
distance_range = max_distance - min_distance

min_time = np.min(time_matrix)
max_time = np.max(time_matrix)
time_range = max_time - min_time

for i in range(nodes):
  for j in range(nodes):
    D[i][j] = (distance_matrix[i][j] - min_distance) / distance_range
    T[i][j] = (time_matrix[i][j] - min_time) / time_range

## Generate Q-Table

Something important to be aware of is the concept of global vs local indexing for tracts.

We have already had global indexing in census tracts, where each GEOID is assigned an integer 0, 1, 2, ...

Local indexing is where the global indexing is given an index that corresponds to its row/column in its local Q table. I have an example below to demonstrate this:

```
global_indexing = { 'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4}
local_indexing = {0: 1, 1: 1, 2: 2, 3: 0, 4: 1}
```

In the example above, this indicates that census tracts `'a': 0`, `'b': 1`, and `'c': 2` are all in the same group, and their local indexes in the Q table are *0 - 2*. Census tracts `'d': 3` and `'e': 4` must be in their own group, where their local indexes in their own Q table is *0 - 1*.

There is also a separate Q table for super-spots, so local indexes had to be made for these as well. In our example, this would appear as below:

```
local_super_indexing = {1: 0, 3: 1}
```

In the example above, this indicates that census tracts `'b': 1` and `'d': 3` are the super-spots in this dataset. Their indexes in their own local Q table are *0 - 1*.

<hr>

In the code, I named the local index dictionary `index_to_Q` (`index_to_Q_super` for the super-spots). I also had dictionaries that reverted local indexing back to global indexing, called `Q_to_index`/`Q_super_to_index`. In the case of `Q_to_index`, to address the issue of repeating local indexes between Q tables, I had to make a multilayered dictionary. In our example, it would look as seen below:

```
Q_to_index = {
  1: {0: 0, 1: 2, 2: 1},
  3: {0: 4, 1: 3}
}
```

In the example above, I first add the super-spots as keys (with their global indexing), then as the values, I add the reversed dictionaries. Notice that the same key used to obtain a Q table of a group, can also be used to obtain the reversed indexes of that group.

This should hopefully make utilizing local/global keys a little less confusing.

I also have two other dictionaries, `tract_group_lists` and `tract_super_dict`.

- `tract_group_lists` is a dictionary where the key is a global super-spot index, and the value is a list of the global hotspot indexes within its group, **including the super spot**. This dictionary is useful several times in Q table generation.

- `tract_super_dict` is a dictionary where the keys are global tract indexes, and the values are the global super-spot index of each tract. It is primarily used in the path simulator to determine which super-spot to travel to next.

In [ ]:
def convert_str_to_int(d):
  return {int(k) if k.isdigit() else k: int(v) if isinstance(v, str) and v.isdigit() else v for k, v in d.items()}

In [ ]:
Q_tables = dict()  # Stores all the Q tables

index_to_Q = dict()  # Local keys for hotspot groups
Q_to_index = dict()  # Inversion of local keys
index_to_Q_super = dict()  # Local keys for superspots
Q_super_to_index = dict()  # Inversion of local keys for superspots

tract_group_lists = dict()  # Used as a reference later in the code, not relevant to the Q Table right now.


# Create Q-Table for every group, with key = superspot idx
for super in gh_df[gh_df['is_super']]['GEOID']:
  # Get the list of all nodes in the group
  group_nodes = list(gh_df[gh_df['GEOID'] == super]['children'])[0].copy()
  group_nodes.append(super)
  tract_group_lists[tract_to_index[super]] = [tract_to_index[n] for n in group_nodes]

  Q_to_index[tract_to_index[super]] = {}
  # Key value pairs for new indices within Q tables
  for idx, n in enumerate(group_nodes):
    index_to_Q[tract_to_index[n]] = idx
    Q_to_index[tract_to_index[super]][idx] = tract_to_index[n]

  # Add Q-table with appropriate key
  Q_tables[tract_to_index[super]] =  np.zeros((len(group_nodes),len(group_nodes),len(group_nodes)))  # np.random.rand(len(group_nodes),len(group_nodes),len(group_nodes))

# Create Q-table for super-spots, with key = -1
super_group = gh_df[gh_df['is_super']]['GEOID'].tolist().copy()
Q_tables[-1] = np.zeros((len(super_group),len(super_group),len(super_group)))  # np.random.rand(len(super_group),len(super_group),len(super_group))

# Key value pairs for new indices within Q tables
tract_group_lists[-1] = [tract_to_index[n] for n in super_group]
for idx, n in enumerate(super_group):
  index_to_Q_super[tract_to_index[n]] = idx
  Q_super_to_index[idx] = tract_to_index[n]


tract_super_dict = {element: key for key, values in tract_group_lists.items() for element in values if key != -1}  # Dictionary that contains what the super-spot of every hotspot

Saves the dictionaries to JSON files

In [ ]:
try:
  os.mkdir(data_dir + f"/Q Tables/")
except:
  print(data_dir + f"/Q Tables/" + " Already Exists")

try:
  os.mkdir(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/")
except:
  print(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/" + " Already Exists")

dict_list_save = ['index_to_Q', 'Q_to_index', 'index_to_Q_super', 'Q_super_to_index', 'tract_group_lists', 'tract_super_dict']

for d in dict_list_save:
  with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/{d}.json", 'w') as f:
    json.dump(globals()[d], f)

Or load them if you want to update the Q-Table

In [ ]:
with open(data_dir + "/Hotspot Data/map_geoid_index.json", 'r') as f:
    tract_to_index = json.load(f)
index_to_tract = {v: k for k, v in tract_to_index.items()}

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/index_to_Q.json", 'r') as f:
  index_to_Q = convert_str_to_int(json.load(f))

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/Q_to_index.json", 'r') as f:
  Q_to_index = convert_str_to_int(json.load(f))

  for k, v in Q_to_index.items():
    Q_to_index[k] = convert_str_to_int(v)

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/index_to_Q_super.json", 'r') as f:
  index_to_Q_super = convert_str_to_int(json.load(f))

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/Q_super_to_index.json", 'r') as f:
  Q_super_to_index = convert_str_to_int(json.load(f))

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/tract_group_lists.json", 'r') as f:
  tract_group_lists = convert_str_to_int(json.load(f))
  tract_group_lists[-1] = tract_group_lists['-1']
  del tract_group_lists['-1']

with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/tract_super_dict.json", 'r') as f:
  tract_super_dict = convert_str_to_int(json.load(f))

Q_tables = np.load(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/Q_boltzman.npz", allow_pickle=True)
Q_tables = {int(key): Q_tables[key] for key in Q_tables}

Functions used in Q-Table generation

In [ ]:
alpha = 0.1
gamma = 0.99
epsilon = 0.99  # 0.8
delta = 0.2
temperature = 1000

def update_q_table(Q, prev_state, action, reward, next_state, terminal_state, alpha, gamma, nodes):
  Qa = min([Q[terminal_state][next_state][a] for a in range(nodes)])
  # https://huggingface.co/blog/assets/73_deep_rl_q_part2/Q-learning-8.jpg
  Q[terminal_state][prev_state][action] += alpha * (reward + gamma * Qa - Q[terminal_state][prev_state][action])

# def epsilon_greedy_policy(Q, terminal_state, state, epsilon, nodes):
#  if random.uniform(0,1) < epsilon:
#   return random.randint(0,nodes-1)
#  else:
#   return min(list(range(nodes)), key = lambda x: Q[terminal_state][(state,x)])

def boltzman_policy(Q, state, temperature, terminal_state, nodes):
  probabilities = np.exp(Q[terminal_state][state] / temperature)
  probabilities /= np.sum(probabilities)

  action = np.random.choice(range(nodes), p=probabilities)
  return action

# In this implementation positive means worse actually, strange I know...
def get_reward(state, action, terminal_state, k):
  if state == action:
    r = +10
  else:
    idx_reverter = Q_to_index[k] if k != -1 else Q_super_to_index
    r = (delta * D[idx_reverter[state]][idx_reverter[action]]) + ((1 - delta) * T[idx_reverter[state]][idx_reverter[action]])

    # Penalty for moving away from final destination
    if D[idx_reverter[action]][idx_reverter[terminal_state]] > D[idx_reverter[state]][idx_reverter[terminal_state]]:  # If the distance from next action to final destination is greater than current location to final destination, worsen reward
      r += 0.1

    if action == terminal_state:
      r += -0.1  # Might need to use -0.75 for chicago, usually -0.1
  return r

def calculate_error(Q, errors, action, terminal_state, nodes, k):
  error = 0
  for node in range(nodes):
    next_state = min(list(range(nodes)), key = lambda x: Q[terminal_state][(node,x)])
    reward = get_reward(node, next_state, terminal_state, k)
    Qa = min([Q[terminal_state][next_state][a] for a in range(nodes)])
    error += (reward + gamma * Qa - Q[terminal_state][node][action])
  errors.append(error)

def gen_Q_Table(episodes=10000, groups=[], anti_groups=[], paths=[]):
  num_episodes = episodes
  errors = list()
  rewards = dict()

  checked_k = anti_groups.copy()

  for k, Q in Q_tables.items():
    if (groups and anti_groups) or (anti_groups and paths):
      raise Exception("Do not specify groups and anti_groups at the same time.")

    # Only generates tables for specified indices
    if groups:
      if k not in groups:
        continue

    # Skips generating tables for specified indices
    if anti_groups:
      if k in anti_groups:
        continue

    selected_Q_nodes = Q.shape[0]

    if not groups:
      print("Checked so far:", checked_k)
    print("Table: ", k, f"[{selected_Q_nodes} nodes]", end="\t\t||\n")

    rewards[k] = list()

    node_range = range(max(paths), selected_Q_nodes) if paths else range(selected_Q_nodes)
    for terminal_state in node_range:

      print("Pathing to node ", terminal_state, end=":\t\t||\t")

      for episode in tqdm(range(num_episodes)):
        r = 0

        prev_state = random.randint(0,selected_Q_nodes-1)
        done = False

        while not done:
          # action = epsilon_greedy_policy(Q, terminal_state, prev_state, epsilon, selected_Q_nodes)
          action = boltzman_policy(Q, prev_state, temperature, terminal_state, selected_Q_nodes)
          reward = get_reward(prev_state, action, terminal_state, k)

          next_state = action
          done = (next_state == terminal_state)

          r += reward
          update_q_table(Q, prev_state, action, reward, next_state, terminal_state, alpha, gamma, selected_Q_nodes)
          prev_state = next_state

        if episode % 100 == 0:
          calculate_error(Q, errors, action, terminal_state, selected_Q_nodes, k)

        rewards[k].append(r)

      # Save this iteration of the Q table
      Q_tables_str_keys = {str(key): value for key, value in Q_tables.items()}  # To avoid a weird issue where numpy doesnt let you save dicts unless the keys are str
      np.savez(f"{data_dir}/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/Q_boltzman.npz", **Q_tables_str_keys)

    checked_k.append(k)
    print()


  # Plot errors
  print("\n")
  plt.scatter(range(len(errors)), errors)
  plt.ylabel("Error")
  plt.xlabel("Episode")
  plt.show()


  with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/errors.json", 'w') as f:
    json.dump(errors, f)


  with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/rewards.json", 'w') as f:
    json.dump(rewards, f)


  # Final save for good measure
  Q_tables_str_keys = {str(key): value for key, value in Q_tables.items()}  # To avoid a weird issue where numpy doesnt let you save dicts unless the keys are str
  np.savez(f"{data_dir}/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/Q_boltzman.npz", **Q_tables_str_keys)

  print("\n\nDONE!")

### Testing Environment

In [ ]:
Q = Q_tables[7]
p = 0   # prev state
start = p
t = 16  # terminal state
nodes = Q.shape[0]

In [ ]:
a = boltzman_policy(Q, p, temperature, t, nodes)
reward = get_reward(p, a, t, k)

n = a
done = (n == t)

Qa = min([Q[t, n, a] for a in range(nodes)])
Q[t, p, a] += alpha * (reward + gamma * Qa - Q[t, p, a])
p = n

print(p)
dict(zip(range(nodes), Q[t, p]))

In [ ]:
multi_iteration_results = []

In [ ]:
k = 7
Q = Q_tables[k]
nodes = Q.shape[0]
for start in tqdm(range(nodes)):  # range(nodes)
  for end in range(nodes):
    for _ in range(100):  # The episodes
      p = start   # prev state
      t = end  # terminal state

      hops = 0
      done = False
      while not done:
        a = boltzman_policy(Q, p, temperature, t, nodes)
        reward = get_reward(p, a, t, k)

        n = a
        done = (n == t)

        Qa = min([Q[t, n, a] for a in range(nodes)])
        Q[t, p, a] += alpha * (reward + gamma * Qa - Q[t, p, a])
        p = n

        hops += 1

      # print(hops)
      multi_iteration_results.append(hops)
      # dict(zip(range(nodes), Q[t, start]))

In [ ]:
dict(zip(range(nodes), Q[15, 22]))

In [ ]:
plt.scatter(range(int(len(multi_iteration_results)/100)), multi_iteration_results[0::100])

### Calling the function

In [ ]:
# checked = [2, 5, 17, 28, 31, 43, 53, 56, 77, 80, 94, 96, 97, 99, 111, 112, 119, 120, 123, 130, 135, 145, 149, 156, 157, 159, 161, 166, 176, 182, 188, 189, 190, 206, 209, 216, 221, 224, 227, 229, 235, 242, 245, 247, 264, 285, 295, 298, 300, 304, 312, 313, 321, 322, 323, 328, 329, 334, 336, 345, 353, 367, 368, 369, 377, 383, 396, 401, 411, 412, 415, 427, 428, 434, 436, 442, 443, 445, 450, 452, 465, 472, 480, 486, 491, 492, 494, 497, 500, 502, 504, 505, 514, 519, 523, 524, 537, 539, 540, 541, 544, 545, 549, 554, 576, 587, 591, 592, 602, 604, 605, 610, 618, 620, 634, 638, 639, 640, 642, 648, 653, 657, 659, 662, 665, 671, 672, 683, 684, 688, 700, 713, 718, 723, 730, 732, 734, 735, 737, 744, 746, 753, 755, 756, 757, 765, 766, 774, 786, 798, 799, 803, 808, 810, 822, 824, 833, 838, 841, 849, 860]
# checked_paths = [0]
import time
start = time.time()

gen_Q_Table(episodes=10000)

end = time.time()

f = open(data_dir + "/ExecutionTime_" + str(superspot_hotspot_ratio) +  "_31.txt", "w")
f.write(str(start - end))
f.close()

# groups: Only generate the Q-Tables for the specified groups.
# anti_groups: Does not generate the Q-Tables for the specified groups.
# Index for super-spot Q-Table is -1


In [ ]:
with open(data_dir + f"/Q Tables/Data-{min_children}-{superspot_hotspot_ratio}/rewards.json", 'r') as f:
  rewards = convert_str_to_int(json.load(f))

# plt.scatter(range(len(rewards)), rewards)
# plt.ylabel("Reward")
# plt.xlabel("Episode")
# plt.show()

exclude_first = 1000
for key, values in rewards.items():
  values = np.array(values)
  # Compute RMS at each time step (cumulative RMS up to that point)
  rms_over_time = np.sqrt(np.cumsum((-1*values)**2) / (np.arange(len(values)) + 1))[exclude_first:]
  # rms_over_time = np.sqrt((-1*values)**2 / (np.arange(len(values)) + 1))[exclude_first:]
  plt.plot(rms_over_time, label=f'Superspot {key} Group', linewidth=1.5)

plt.xlabel("Episode")
plt.ylabel("RMS")
plt.title(f"RMS of Cumulative Reward Values by Episode (Excluding first {exclude_first})")
# plt.legend()
plt.show()


In [ ]:
print(data_dir)
print(start - end)

## Check Correctness of Q-Table

In [ ]:
def get_second_shortest(Q, k, i, j):
  if i == j:  # If equal return true.
    return True

  # Idx converter
  idx_converter = Q_to_index[k] if k != -1 else Q_super_to_index

  # Path/score lists for distance matrix pathing
  paths = []
  scores = []

  options = distance_matrix[idx_converter[i]]  # Gets the options from the distance matrix based on starting value

  for idx, o in enumerate(options):  # For options and indices...
    if o < sys.maxsize and idx != idx_converter[i] and idx != idx_converter[j] and idx in tract_group_lists[k]:  # If o is not max value (nonexistent edge) and index is not the start and not the end and index is in the same group...

      if o == sys.maxsize: raise Exception("ERROR!!!")  # Exception if edge does not exist somehow
      paths.append([idx_converter[i], idx])  # Add path [<start>, idx] to path list
      scores.append(o)  # Add score to score list

  for score_idx, p in enumerate(paths):  # For p in paths
    o = distance_matrix[p[-1], idx_converter[j]]  # Get distance from that path to the destination
    if o == sys.maxsize: raise Exception("ERROR!!!")  # Raise exception if path somehow does not exist
    scores[score_idx] += o  # Update score for that path
    paths[score_idx].append(idx_converter[j])  # Add destination to path list


  combined = list(zip(scores, paths))  # Zip scores, paths
  combined.sort()  # Sort by score
  second_smallest_score, corresponding_path = combined[0]  # Get second smallest (here index 0 works because I am excluding direct path by making sure that start/end nodes cannot be part of the intermediate steps)


  Q_score = 0  # Score for Q-table method
  Q_path = [i]  # Path for Q-table method

  options = Q[j][i]  # Gets options for destiation j given current position i
  second_smallest_idx = np.argpartition(options, 1)[1]  # Gets index of second smallest value in list

  Q_score += distance_matrix[idx_converter[i], idx_converter[second_smallest_idx]]  # Add score of moving from i to idx using distance matrix
  Q_score += distance_matrix[idx_converter[second_smallest_idx], idx_converter[j]]  # Add score of moving from idx to j using distance matrix

  Q_path.append(second_smallest_idx)  # Append path steps to Q_path
  Q_path.append(j)

  # NOTE: Paths are not returned at all, however they are available to be printed for debugging as necessary.

  # Check if scores are equal
  return second_smallest_score == Q_score

In [ ]:
print("First box shows correctness of direct path, second shows correctness of second-best path.")

def check_within_groups(Q_tables, lim:int=10, show:str="both", groups=[]):
  for k, Q in Q_tables.items():
    if groups:
      if k not in groups:
        continue

    Q_shape = Q.shape[0]
    print("Q-table", k, "\t||", "="*(12*Q_shape+1))
    for j in range(Q_shape):
      print(" -= Paths to", j, end="\t||  ")
      for i in range(Q_shape):
        i_saved = i
        inf_counter = 1

        while i != j:
          if inf_counter > lim: break

          m = 1000000  # Arbitrary large number
          values = np.array(Q[j][i])
          idx = np.argmin(values)
          i, m = idx, values[idx]

          inf_counter += 1

        direct, second = "", ""
        if show == "both" or show == "direct":
          direct = "✅" if inf_counter <= lim else "❌"
        if show == "both" or show == "second":
          second = "✅" if get_second_shortest(Q, k, i_saved, j) else "❌"
        line = str(i_saved) + f" [{direct}|{second}]"
        print(line, end="  |  ")

      print()

check_within_groups(Q_tables, lim=2, show="direct")